# Production-Grade Bandit Engine

**Goal:** Build a complete production bandit system with logging, monitoring, and alerts in 15 minutes.

**What you'll build:**
- BanditEngine class with arm registry, selection, and reward tracking
- Structured JSON logging for every decision
- Real-time monitoring with anomaly detection
- Alert system for policy collapse and reward degradation

In [ ]:
import numpy as np
import json
from datetime import datetime
from collections import defaultdict, deque, Counter
import matplotlib.pyplot as plt

np.random.seed(42)

## Step 1: Production Bandit Engine

Complete system with separation of concerns: arm management, policy logic, logging, monitoring.

In [ ]:
class BanditEngine:
    """Production-grade multi-armed bandit."""

    def __init__(self, epsilon=0.1, window_size=20):
        # Arm registry
        self.arms = {}  # arm_id -> metadata
        self.enabled_arms = set()

        # Policy state (epsilon-greedy Thompson Sampling hybrid)
        self.epsilon = epsilon
        self.alpha = defaultdict(lambda: 1.0)  # Beta prior alpha
        self.beta = defaultdict(lambda: 1.0)   # Beta prior beta

        # Logging
        self.decision_log = []

        # Monitoring
        self.monitor_window = window_size
        self.recent_arms = deque(maxlen=window_size)
        self.recent_rewards = deque(maxlen=window_size)
        self.alerts = []

    def register_arm(self, arm_id, metadata=None):
        """Add an arm to the registry."""
        self.arms[arm_id] = metadata or {}
        self.enabled_arms.add(arm_id)

    def disable_arm(self, arm_id):
        """Disable an arm (circuit breaker)."""
        self.enabled_arms.discard(arm_id)

    def select_arm(self, context=None, decision_id=None):
        """Select arm using Thompson Sampling with epsilon exploration."""
        active = list(self.enabled_arms)
        if not active:
            raise ValueError("No active arms!")

        # Epsilon-greedy exploration
        if np.random.random() < self.epsilon:
            selected = np.random.choice(active)
            exploration_type = "random"
        else:
            # Thompson Sampling
            samples = {arm: np.random.beta(self.alpha[arm], self.beta[arm])
                      for arm in active}
            selected = max(samples, key=samples.get)
            exploration_type = "thompson"

        # Log decision
        log_entry = {
            "timestamp": datetime.now().isoformat(),
            "decision_id": decision_id or len(self.decision_log),
            "policy_version": "thompson_epsilon_v1.0",
            "context": context or {},
            "active_arms": active,
            "selected_arm": selected,
            "exploration_type": exploration_type,
            "arm_stats": {arm: {"alpha": self.alpha[arm], "beta": self.beta[arm]}
                         for arm in active}
        }
        self.decision_log.append(log_entry)

        return selected

    def record_reward(self, arm, reward):
        """Update policy with observed reward."""
        # Update Thompson Sampling priors
        self.alpha[arm] += reward
        self.beta[arm] += (1 - reward)

        # Update monitoring
        self.recent_arms.append(arm)
        self.recent_rewards.append(reward)

        # Log reward
        if self.decision_log:
            self.decision_log[-1]["reward"] = reward
            self.decision_log[-1]["reward_timestamp"] = datetime.now().isoformat()

        # Check alerts
        self._check_alerts()

    def get_stats(self):
        """Get current system statistics."""
        stats = {
            "total_decisions": len(self.decision_log),
            "active_arms": list(self.enabled_arms),
            "arm_means": {arm: self.alpha[arm] / (self.alpha[arm] + self.beta[arm])
                         for arm in self.enabled_arms},
            "arm_pull_counts": dict(Counter(e["selected_arm"] for e in self.decision_log)),
        }

        if len(self.recent_arms) >= 10:
            # Selection entropy
            counts = Counter(self.recent_arms)
            probs = np.array([counts[a] for a in counts]) / len(self.recent_arms)
            entropy = -np.sum(probs * np.log(probs + 1e-10))
            stats["selection_entropy"] = entropy

            # Recent performance
            stats["recent_avg_reward"] = np.mean(self.recent_rewards)
            stats["recent_std_reward"] = np.std(self.recent_rewards)

        return stats

    def _check_alerts(self):
        """Check for anomalies and generate alerts."""
        if len(self.recent_arms) < self.monitor_window:
            return

        # Alert 1: Policy collapse (entropy too low)
        counts = Counter(self.recent_arms)
        probs = np.array([counts[a] for a in counts]) / len(self.recent_arms)
        entropy = -np.sum(probs * np.log(probs + 1e-10))

        if entropy < 0.5:
            alert = f"⚠️  POLICY COLLAPSE: Entropy={entropy:.3f} < 0.5"
            if alert not in self.alerts[-5:]:  # Avoid spam
                self.alerts.append(alert)
                print(alert)

        # Alert 2: Reward degradation (using simple threshold)
        recent_mean = np.mean(self.recent_rewards)
        expected_baseline = 0.5  # For Bernoulli rewards

        if recent_mean < 0.3:  # Below 30% success rate
            alert = f"⚠️  REWARD DEGRADATION: {recent_mean:.3f} < 0.3"
            if alert not in self.alerts[-5:]:
                self.alerts.append(alert)
                print(alert)

print("✓ BanditEngine class ready")

## Step 2: Run 1000 Rounds with Full Logging

In [ ]:
# Initialize engine
engine = BanditEngine(epsilon=0.1, window_size=50)

# Register arms (simulating commodities with different success rates)
true_means = {
    "GOLD": 0.55,   # Slightly better
    "OIL": 0.45,
    "NATGAS": 0.40,
    "COPPER": 0.50
}

for arm in true_means:
    engine.register_arm(arm, metadata={"true_mean": true_means[arm]})

# Run 1000 rounds
rewards_history = []
regrets = []
best_mean = max(true_means.values())

for t in range(1000):
    # Select arm
    arm = engine.select_arm(context={"round": t}, decision_id=f"round_{t}")

    # Simulate reward (Bernoulli with true mean)
    reward = 1.0 if np.random.random() < true_means[arm] else 0.0

    # Record reward
    engine.record_reward(arm, reward)

    # Track metrics
    rewards_history.append(reward)
    regret = best_mean - true_means[arm]
    regrets.append(regret)

print(f"\n✓ Completed 1000 rounds")
print(f"Total alerts raised: {len(engine.alerts)}")
print(f"Cumulative regret: {sum(regrets):.2f}")

## Step 3: Monitoring Summary

Analyze arm selection frequencies, reward trends, estimated regret.

In [ ]:
stats = engine.get_stats()

print("=" * 60)
print("PRODUCTION MONITORING DASHBOARD")
print("=" * 60)

print(f"\n📊 OVERALL STATS")
print(f"   Total decisions: {stats['total_decisions']}")
print(f"   Active arms: {', '.join(stats['active_arms'])}")

print(f"\n🎯 ARM SELECTION DISTRIBUTION")
for arm, count in sorted(stats['arm_pull_counts'].items(), key=lambda x: -x[1]):
    pct = 100 * count / stats['total_decisions']
    bar = "█" * int(pct / 2)
    print(f"   {arm:8s} {bar:20s} {pct:5.1f}% ({count} pulls)")

print(f"\n📈 LEARNED ARM MEANS (vs True)")
for arm in stats['active_arms']:
    learned = stats['arm_means'][arm]
    true = true_means[arm]
    error = abs(learned - true)
    print(f"   {arm:8s} Learned: {learned:.3f}  True: {true:.3f}  Error: {error:.3f}")

print(f"\n⚡ RECENT PERFORMANCE (last {engine.monitor_window} rounds)")
print(f"   Avg reward: {stats['recent_avg_reward']:.3f} ± {stats['recent_std_reward']:.3f}")
print(f"   Selection entropy: {stats['selection_entropy']:.3f}")
if stats['selection_entropy'] < 0.5:
    print("   ⚠️  Low entropy - policy may have collapsed!")
elif stats['selection_entropy'] > 1.5:
    print("   ⚠️  High entropy - still exploring heavily")
else:
    print("   ✓ Healthy exploration/exploitation balance")

print(f"\n🚨 ALERTS HISTORY")
if engine.alerts:
    for alert in engine.alerts[-5:]:
        print(f"   {alert}")
else:
    print("   ✓ No alerts raised")

print("\n" + "=" * 60)

## Step 4: Visualize Key Metrics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Cumulative regret
axes[0, 0].plot(np.cumsum(regrets), linewidth=2)
axes[0, 0].set_title("Cumulative Regret Over Time", fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel("Round")
axes[0, 0].set_ylabel("Cumulative Regret")
axes[0, 0].grid(alpha=0.3)

# Arm selection distribution
arm_counts = stats['arm_pull_counts']
axes[0, 1].bar(arm_counts.keys(), arm_counts.values(), color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A'])
axes[0, 1].set_title("Arm Selection Frequency", fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel("Arm")
axes[0, 1].set_ylabel("Number of Pulls")
axes[0, 1].grid(axis='y', alpha=0.3)

# Reward moving average
window = 50
moving_avg = np.convolve(rewards_history, np.ones(window)/window, mode='valid')
axes[1, 0].plot(moving_avg, linewidth=2, label=f'{window}-round MA')
axes[1, 0].axhline(best_mean, color='green', linestyle='--', alpha=0.7, label='Best arm mean')
axes[1, 0].set_title("Reward Moving Average", fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel("Round")
axes[1, 0].set_ylabel("Avg Reward")
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Arm selection over time (last 200 rounds)
recent_decisions = [log['selected_arm'] for log in engine.decision_log[-200:]]
arm_to_num = {arm: i for i, arm in enumerate(sorted(true_means.keys()))}
arm_nums = [arm_to_num[arm] for arm in recent_decisions]
axes[1, 1].scatter(range(len(arm_nums)), arm_nums, alpha=0.5, s=10)
axes[1, 1].set_title("Arm Selection Pattern (Last 200 Rounds)", fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel("Round")
axes[1, 1].set_ylabel("Arm")
axes[1, 1].set_yticks(range(len(true_means)))
axes[1, 1].set_yticklabels(sorted(true_means.keys()))
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Monitoring visualizations complete")

## Step 5: Inspect Structured Logs

Show the power of comprehensive logging for debugging.

In [ ]:
print("Sample decision logs (first 3 and last 3):\n")

for i in [0, 1, 2, -3, -2, -1]:
    log_entry = engine.decision_log[i]
    print(f"Decision {log_entry['decision_id']}:")
    print(f"  Timestamp: {log_entry['timestamp']}")
    print(f"  Selected: {log_entry['selected_arm']} ({log_entry['exploration_type']})")
    print(f"  Reward: {log_entry.get('reward', 'N/A')}")
    print(f"  Active arms: {log_entry['active_arms']}")
    print()

# Export to JSON file (in production, this would go to S3/GCS)
print("Exporting full logs to JSON...")
with open('/tmp/bandit_decisions.jsonl', 'w') as f:
    for entry in engine.decision_log:
        f.write(json.dumps(entry) + '\n')

print(f"✓ Exported {len(engine.decision_log)} decision logs to /tmp/bandit_decisions.jsonl")

## Step 6: Detect Policy Collapse

Simulate a scenario where the bandit gets stuck on one arm.

In [ ]:
print("Simulating policy collapse scenario...\n")

# Create new engine with very low exploration
collapse_engine = BanditEngine(epsilon=0.01, window_size=30)

# Register arms with one clearly dominant arm
collapse_means = {"ARM_A": 0.7, "ARM_B": 0.3, "ARM_C": 0.3, "ARM_D": 0.3}
for arm in collapse_means:
    collapse_engine.register_arm(arm)

# Run until policy collapses
for t in range(200):
    arm = collapse_engine.select_arm()
    reward = 1.0 if np.random.random() < collapse_means[arm] else 0.0
    collapse_engine.record_reward(arm, reward)

# Check final stats
collapse_stats = collapse_engine.get_stats()

print("\n📊 POLICY COLLAPSE DETECTION")
print(f"   Selection entropy: {collapse_stats['selection_entropy']:.3f}")
print(f"   Arm distribution:")
for arm, count in sorted(collapse_stats['arm_pull_counts'].items(), key=lambda x: -x[1]):
    pct = 100 * count / collapse_stats['total_decisions']
    print(f"      {arm}: {pct:.1f}%")

if collapse_stats['selection_entropy'] < 0.5:
    print("\n⚠️  ALERT: Policy has collapsed! Almost always picking ARM_A.")
    print("   Action: Increase epsilon or reset policy with optimistic initialization.")

## Modify This: Experiment with Production Features

Try these modifications:
1. **Change alert thresholds** — Make entropy threshold more/less sensitive
2. **Add custom guardrails** — Prevent selecting same arm more than 10 times in a row
3. **Export to different format** — Save logs to CSV instead of JSONL
4. **Add more monitoring metrics** — Track maximum consecutive selections of same arm
5. **Implement circuit breaker** — Auto-disable arms with reward < 0.2 for last 20 pulls

In [ ]:
# Your experiments here

# Example: Add circuit breaker
def check_circuit_breaker(engine, threshold=0.2, min_pulls=20):
    """Disable arms performing below threshold."""
    # Your implementation
    pass

## Summary

**What you built:**
- Complete production bandit engine with arm registry, policy, logging, monitoring
- Structured JSON logging capturing every decision with full context
- Real-time monitoring with entropy and reward degradation detection
- Alert system that catches policy collapse before it causes losses

**Key production principles:**
1. **Separation of concerns** — Policy, logging, monitoring are independent
2. **Comprehensive logging** — Capture everything for post-hoc debugging
3. **Actionable alerts** — Don't just track metrics, define failure thresholds
4. **Monitoring dashboard** — Real-time visibility into system health

**Next steps:**
- Notebook 02: Migrate from A/B testing to bandits
- Notebook 03: Deploy end-to-end commodity allocation system
- Read guide on offline evaluation for testing new policies on logged data